# Week 5: Data Science Automation

**Wesley Garrett**
MSDS 600
Due: 2026-09-29


## Setup

Kernel: **MSDS Week 5 (pycaret)**. Relative paths only. No live `%pip install`.


In [1]:
from pathlib import Path
import pandas as pd

# Training data from week 2 cleaning
TRAIN_PATH = Path("../../../../week_02/data/week_02_data_churn_cleaned.csv")
# New customers to score (LMS name: new_churn_data.csv)
NEW_PATH = Path("../../../data/week_05_data_new_churn.csv")
# Same customers with Yes/No text categories (helps match week 2 columns)
UNMOD_PATH = Path("../../../data/week_05_data_new_churn_unmodified.csv")

print("Train exists:", TRAIN_PATH.exists(), TRAIN_PATH)
print("New exists:", NEW_PATH.exists(), NEW_PATH)
print("Unmod exists:", UNMOD_PATH.exists(), UNMOD_PATH)

train = pd.read_csv(TRAIN_PATH)
print("Train shape:", train.shape)
print("Churn rate:", round(float(train["Churn_label"].mean()), 4))
train.head()


Train exists: True ../../../../week_02/data/week_02_data_churn_cleaned.csv
New exists: True ../../../data/week_05_data_new_churn.csv
Unmod exists: True ../../../data/week_05_data_new_churn_unmodified.csv
Train shape: (7032, 13)
Churn rate: 0.2658


,customerID,tenure,MonthlyCharges,TotalCharges,Churn,Churn_label,PhoneService_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,avg_charge_per_tenure
0,7590-VHVEG,1,29.85,29.85,No,0,False,False,False,False,True,False,29.850000
1,5575-GNVDE,34,56.95,1889.50,No,0,True,True,False,False,False,True,55.573529
2,3668-QPYBK,2,53.85,108.15,Yes,1,True,False,False,False,False,True,54.075000
3,7795-CFOCW,45,42.30,1840.75,No,0,False,True,False,False,False,False,40.905556
4,9237-HQITU,2,70.70,151.65,Yes,1,True,False,False,False,True,False,75.825000


## Task 1: AutoML with pycaret (churn)

I used the week 2 cleaned churn file. Only about 27% of customers leave, so accuracy alone can look good just by guessing "stays." I asked pycaret to rank models by **AUC**, which is better at ranking likely leavers. The week 5 FTE shows `compare_models(sort=...)` for picking a metric; week 3 covered why AUC / precision / recall can matter more than accuracy on uneven classes.

Plan:
1. Let pycaret compare models
2. Save the best one
3. Score the five new customers with `predict_churn.py`

The new CSV uses different column names and encodings than week 2 (`charge_per_tenure` vs `avg_charge_per_tenure`). In the Python file I fix that before scoring.

Note: the week 2 training set still has both `MonthlyCharges` and `avg_charge_per_tenure`. Those can overlap (average charge is related to the monthly bill), but I kept the week 2 prepared columns as the starter asked.


In [2]:
from pycaret.classification import (
    setup,
    compare_models,
    pull,
    save_model,
    load_model,
    predict_model,
)

# Keep Churn_label as the target; drop id and the Yes/No text column
model_df = train.drop(columns=["customerID", "Churn"]).copy()

automl = setup(
    data=model_df,
    target="Churn_label",
    session_id=42,
    verbose=False,
    html=False,
)


In [3]:
# Why exclude these models?
# Week 5 FTE says boosting models like xgboost and catboost take the longest in compare_models.
# I also skip lightgbm (same booster family) and gpc (slow Gaussian process).
# This is a runtime choice on this server, not a claim that those models are worse.
# I still sort the remaining models by AUC.
best_model = compare_models(
    sort="AUC",
    exclude=["xgboost", "catboost", "lightgbm", "gpc"],
)
results = pull()
print("Best model:")
print(best_model)
results.head(10)


Processing:   0%|          | 0/45 [00:00<?, ?it/s]

Processing:  11%|█         | 5/45 [00:16<02:11,  3.29s/it]

Processing:  20%|██        | 9/45 [00:26<01:44,  2.90s/it]

Processing:  24%|██▍       | 11/45 [00:27<01:15,  2.22s/it]

Processing:  29%|██▉       | 13/45 [00:28<00:53,  1.66s/it]

Processing:  33%|███▎      | 15/45 [00:28<00:36,  1.22s/it]

... (13 outputs left out)

Processing:  96%|█████████▌| 43/45 [00:49<00:02,  1.27s/it]

Processing: 100%|██████████| 45/45 [00:49<00:00,  1.04it/s]

                                 Model  Accuracy     AUC  Recall   Prec.  \
gbc       Gradient Boosting Classifier    0.7942  0.8393  0.4962  0.6486   
lr                 Logistic Regression    0.7944  0.8375  0.5252  0.6387   
ada               Ada Boost Classifier    0.7926  0.8349  0.5176  0.6375   
ridge                 Ridge Classifier    0.7901  0.8258  0.4617  0.6493   
lda       Linear Discriminant Analysis    0.7895  0.8258  0.5267  0.6235   
qda    Quadratic Discriminant Analysis    0.7326  0.8239  0.7721  0.4985   
nb                         Naive Bayes    0.7316  0.8143  0.7400  0.4966   
rf            Random Forest Classifier    0.7765  0.8056  0.4725  0.6029   
et              Extra Trees Classifier    0.7653  0.7894  0.4839  0.5689   
knn             K Neighbors Classifier    0.7595  0.7457  0.4282  0.5627   
svm                SVM - Linear Kernel    0.7058  0.6849  0.5248  0.5683   
dt            Decision Tree Classifier    0.7310  0.6619  0.5008  0.4946   
dummy       

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.7942,0.8393,0.4962,0.6486,0.5615,0.4302,0.4373,0.747
lr,Logistic Regression,0.7944,0.8375,0.5252,0.6387,0.5758,0.4419,0.4460,1.643
ada,Ada Boost Classifier,0.7926,0.8349,0.5176,0.6375,0.5699,0.4353,0.4403,0.307
ridge,Ridge Classifier,0.7901,0.8258,0.4617,0.6493,0.5388,0.4080,0.4185,0.040
lda,Linear Discriminant Analysis,0.7895,0.8258,0.5267,0.6235,0.5705,0.4325,0.4356,0.048
qda,Quadratic Discriminant Analysis,0.7326,0.8239,0.7721,0.4985,0.6053,0.4172,0.4407,0.058
nb,Naive Bayes,0.7316,0.8143,0.7400,0.4966,0.5941,0.4050,0.4233,0.046
rf,Random Forest Classifier,0.7765,0.8056,0.4725,0.6029,0.5288,0.3853,0.3908,0.225
et,Extra Trees Classifier,0.7653,0.7894,0.4839,0.5689,0.5224,0.3684,0.3708,0.195
knn,K Neighbors Classifier,0.7595,0.7457,0.4282,0.5627,0.4858,0.3327,0.3383,1.016


In [4]:
# Save next to this notebook so predict_churn.py can load it
save_model(best_model, "churn_pycaret_model")
print("Saved churn_pycaret_model.pkl")


Transformation Pipeline and Model Successfully Saved
Saved churn_pycaret_model.pkl


### Python module

`predict_churn.py` loads the saved model and returns a churn probability for each row.


In [5]:
from predict_churn import load_data, predict_churn_proba, align_features_like_week2

new_df = load_data(NEW_PATH)
print("New data:")
display(new_df)

print("Aligned features (week-2 style):")
display(align_features_like_week2(new_df))

probs = predict_churn_proba(new_df)
print("Churn probabilities:")
print(probs)

# Optional labels from the starter (for a quick self-check)
true_labels = [1, 0, 0, 1, 0]
pred_labels = (probs >= 0.5).astype(int).tolist()
print("True labels (starter):", true_labels)
print("Hard labels at 0.5:", pred_labels)
print("Matches:", [int(a == b) for a, b in zip(pred_labels, true_labels)])


New data:


,customerID,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure
0,9305-CKSKC,22,1,0,2,97.40,811.70,36.895455
1,1452-KNGVK,8,0,1,1,77.30,1701.95,212.743750
2,6723-OKKJM,28,1,0,0,28.25,250.90,8.960714
3,7832-POPKP,62,1,0,2,101.70,3106.56,50.105806
4,6348-TACGU,10,0,0,1,51.15,3440.97,344.097000


Aligned features (week-2 style):


,tenure,MonthlyCharges,TotalCharges,PhoneService_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,avg_charge_per_tenure
0,22,97.40,811.70,True,False,False,False,True,False,36.895455
1,8,77.30,1701.95,False,True,False,False,False,True,212.743750
2,28,28.25,250.90,True,False,False,True,False,False,8.960714
3,62,101.70,3106.56,True,False,False,False,True,False,50.105806
4,10,51.15,3440.97,False,False,True,True,False,False,344.097000


Transformation Pipeline and Model Successfully Loaded


Churn probabilities:
0    0.4092
1    0.4367
2    0.1023
3    0.2510
4    0.2497
Name: churn_probability, dtype: float64
True labels (starter): [1, 0, 0, 1, 0]
Hard labels at 0.5: [0, 0, 0, 0, 0]
Matches: [0, 1, 1, 0, 1]


In [6]:
# Run the module like a normal Python script
%run predict_churn.py


Loading: ../../../data/week_05_data_new_churn.csv
Rows: 5
Columns: ['customerID', 'tenure', 'PhoneService', 'Contract', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'charge_per_tenure']
Transformation Pipeline and Model Successfully Loaded
Churn probabilities:
0    0.4092
1    0.4367
2    0.1023
3    0.2510
4    0.2497
Name: churn_probability, dtype: float64


## Summary


**Your answer:**

I trained pycaret on the week 2 cleaned churn file (7,032 customers). I ranked models by **AUC** because churn is uneven, most people stay, so accuracy can look high even when the model misses leavers.

The best model was a **Gradient Boosting Classifier** (about 0.839 AUC). Logistic regression and AdaBoost were close behind. I excluded xgboost, catboost, lightgbm, and gpc because the week 5 FTE notes that boosters like xgboost and catboost take the longest, and those runs were too slow on this server. That was a time tradeoff, not a judgment that those models cannot work.

I saved the model as `churn_pycaret_model.pkl` and wrote `predict_churn.py` so a function can take a DataFrame and return churn probability per row. The new CSV did not match the week 2 columns, so the script aligns those columns first.

On the five new customers, the probabilities were about 0.41, 0.44, 0.10, 0.25, and 0.25. The usual rule is to call someone a churner if that probability is 0.5 or higher. All five were under 0.5, so none were labeled as churn. The starter's optional true labels are `[1, 0, 0, 1, 0]`, so after autoML picks a model I still have to choose where to draw that stay-vs-churn line.


## Task 2: Critical Reflection

What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?


**Your answer:**

Pycaret made it easy to compare many models on the same week 2 churn table. Gradient boosting won on AUC, with logistic regression close behind. When scores are close, the top pick can change a little between runs.

For churn, I care more about finding leavers than getting the highest accuracy on a mostly "stay" group. That is why I used AUC. Even the best model only caught about half of leavers in the comparison table, so I would still think about where to draw the stay-vs-churn line (for example, calling someone a churner at 0.4 instead of 0.5 if missing leavers costs more).

Even a strong leaderboard model needs the new customers in the same column layout as week 2. The new CSV used different column names, so `predict_churn.py` rebuilds those columns before scoring.

So gradient boosting was useful here, but the bigger win for me was the full path: choose a metric that fits the goal, save the model, and put prediction in a small Python file.


## AI Use Disclosure


**Your answer:**

I used generative AI for research help, to organize this notebook and `predict_churn.py`, and to set up the Week 5 pycaret kernel without changing my main MSDS (Regis) environment. I checked the results myself by running the notebook on the week 2 churn CSV and `new_churn_data.csv`, and I used it to help compare the model metrics before I chose the best model.


## GitHub / submission

I will upload this notebook and `predict_churn.py` to GitHub and turn in the repo link in the week 5 dropbox.

GitHub link: https://github.com/wesgarr/msds600-week5-churn-automation
